# Satellite Change Detection — Siamese U-Net Training

This notebook trains a **Siamese U-Net** on the **LEVIR-CD+** dataset for pixel-level
satellite image change detection. The exported model plugs directly into the
AI Change Detection web app.

**Optimized for CPU** — uses a lightweight MobileNetV2 encoder and 15 epochs.
Training takes ~3-4 hours on a Colab CPU runtime.

## 1. Install Dependencies

In [ ]:
!pip install -q torch torchvision segmentation-models-pytorch albumentations datasets tqdm matplotlib

## 2. Download & Prepare LEVIR-CD+ Dataset

LEVIR-CD+ contains 985 pairs of 1024×1024 Google Earth images with pixel-level
building change annotations. We download it from **Hugging Face** (reliable CDN,
no Google Drive rate limits), then cut each image into 256×256 patches.

In [ ]:
import os
import numpy as np
from PIL import Image
from datasets import load_dataset

DATA_ROOT = "./levir_cd_256"
PATCH_SIZE = 256

def save_patches(split_data, out_dir, start_idx=0):
    """Cut 1024×1024 images into 256×256 patches and save to disk."""
    os.makedirs(os.path.join(out_dir, "A"), exist_ok=True)
    os.makedirs(os.path.join(out_dir, "B"), exist_ok=True)
    os.makedirs(os.path.join(out_dir, "label"), exist_ok=True)
    patch_id = start_idx
    for row in split_data:
        img_a = np.array(row["image1"].convert("RGB"))
        img_b = np.array(row["image2"].convert("RGB"))
        mask  = np.array(row["mask"].convert("L"))
        h, w = img_a.shape[:2]
        for y in range(0, h - PATCH_SIZE + 1, PATCH_SIZE):
            for x in range(0, w - PATCH_SIZE + 1, PATCH_SIZE):
                pa = img_a[y:y+PATCH_SIZE, x:x+PATCH_SIZE]
                pb = img_b[y:y+PATCH_SIZE, x:x+PATCH_SIZE]
                pm = mask[y:y+PATCH_SIZE, x:x+PATCH_SIZE]
                name = f"{patch_id:05d}.png"
                Image.fromarray(pa).save(os.path.join(out_dir, "A", name))
                Image.fromarray(pb).save(os.path.join(out_dir, "B", name))
                Image.fromarray(pm).save(os.path.join(out_dir, "label", name))
                patch_id += 1
    return patch_id

if not os.path.isdir(DATA_ROOT):
    print("Downloading LEVIR-CD+ from Hugging Face (~3.8 GB)...")
    ds = load_dataset("blanchon/LEVIR_CDPlus")

    # The dataset has 'train' and 'test' splits
    train_data = ds["train"]
    test_data  = ds["test"]

    # Use last 10% of train as validation
    n_train = len(train_data)
    n_val = max(1, int(n_train * 0.1))
    val_indices   = list(range(n_train - n_val, n_train))
    train_indices = list(range(0, n_train - n_val))

    print(f"Total train images: {n_train}, using {len(train_indices)} train + {len(val_indices)} val")
    print(f"Test images: {len(test_data)}")

    print("Cutting into 256×256 patches (this takes a few minutes)...")
    n = save_patches(train_data.select(train_indices), os.path.join(DATA_ROOT, "train"))
    print(f"  Train patches: {n}")
    n = save_patches(train_data.select(val_indices), os.path.join(DATA_ROOT, "val"))
    print(f"  Val patches:   {n}")
    n = save_patches(test_data, os.path.join(DATA_ROOT, "test"))
    print(f"  Test patches:  {n}")
    print("Done! Dataset at:", DATA_ROOT)
else:
    print("Dataset already present at", DATA_ROOT)

In [ ]:
# Verify structure — adjust paths if your zip extracts differently
for split in ["train", "val", "test"]:
    for sub in ["A", "B", "label"]:
        p = os.path.join(DATA_ROOT, split, sub)
        if os.path.isdir(p):
            n = len(os.listdir(p))
            print(f"{split}/{sub}: {n} files")
        else:
            print(f"WARNING: {p} not found — check extracted folder name")

## 3. Dataset & DataLoader

In [ ]:
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2


class LEVIRCDDataset(Dataset):
    """LEVIR-CD patch dataset: before (A), after (B), binary label."""

    def __init__(self, root, split="train", transform=None):
        self.dir_a = os.path.join(root, split, "A")
        self.dir_b = os.path.join(root, split, "B")
        self.dir_label = os.path.join(root, split, "label")
        self.fnames = sorted(os.listdir(self.dir_a))
        self.transform = transform

    def __len__(self):
        return len(self.fnames)

    def __getitem__(self, idx):
        name = self.fnames[idx]
        img_a = np.array(Image.open(os.path.join(self.dir_a, name)).convert("RGB"))
        img_b = np.array(Image.open(os.path.join(self.dir_b, name)).convert("RGB"))
        label = np.array(Image.open(os.path.join(self.dir_label, name)).convert("L"))
        label = (label > 127).astype(np.float32)

        if self.transform:
            aug = self.transform(
                image=img_a,
                image_b=img_b,
                mask=label,
            )
            img_a = aug["image"]        # (3, H, W) tensor
            img_b = aug["image_b"]      # (3, H, W) tensor
            label = aug["mask"].unsqueeze(0)  # (1, H, W)
        return img_a, img_b, label


train_transform = A.Compose(
    [
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.RandomBrightnessContrast(p=0.3, brightness_limit=0.15, contrast_limit=0.15),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ],
    additional_targets={"image_b": "image"},
)

val_transform = A.Compose(
    [
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ],
    additional_targets={"image_b": "image"},
)

train_ds = LEVIRCDDataset(DATA_ROOT, "train", train_transform)
val_ds = LEVIRCDDataset(DATA_ROOT, "val", val_transform)
test_ds = LEVIRCDDataset(DATA_ROOT, "test", val_transform)

BATCH = 4  # smaller batch for CPU
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=0, pin_memory=False)
val_dl = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=False)
test_dl = DataLoader(test_ds, batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=False)

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

## 4. Siamese U-Net Model

Architecture:
- **Shared encoder** (MobileNetV2, ImageNet pretrained) — lightweight and fast on CPU
- Feature maps from both branches are **concatenated** at each decoder level
- Standard U-Net decoder produces a binary change mask

In [ ]:
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp


ENCODER_NAME = "mobilenet_v2"  # lightweight encoder for CPU training


class SiameseUNet(nn.Module):
    """
    Siamese U-Net for change detection.
    Shared encoder extracts features from both images;
    concatenated features are decoded into a binary change mask.
    """

    def __init__(self, encoder_name=ENCODER_NAME, pretrained=True):
        super().__init__()
        aux = smp.Unet(
            encoder_name=encoder_name,
            encoder_weights="imagenet" if pretrained else None,
            in_channels=3,
            classes=1,
        )
        self.encoder = aux.encoder

        encoder_channels = self.encoder.out_channels
        doubled = tuple(c * 2 for c in encoder_channels)

        self.decoder = smp.decoders.unet.decoder.UnetDecoder(
            encoder_channels=doubled,
            decoder_channels=(256, 128, 64, 32, 16),
            n_blocks=5,
            use_batchnorm=True,
            attention_type=None,
        )

        self.head = nn.Conv2d(16, 1, kernel_size=1)

    def forward(self, img_a, img_b):
        feats_a = self.encoder(img_a)
        feats_b = self.encoder(img_b)
        feats_cat = [torch.cat([fa, fb], dim=1) for fa, fb in zip(feats_a, feats_b)]
        decoded = self.decoder(*feats_cat)
        logits = self.head(decoded)
        return logits


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SiameseUNet(encoder_name=ENCODER_NAME, pretrained=True).to(device)

total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Model on {device}, {total_params:.1f}M parameters")
if device.type == "cpu":
    print("Running on CPU — training will take ~3-4 hours for 15 epochs")

## 5. Loss Function & Metrics

Combined **BCE + Dice** loss handles class imbalance (most pixels are unchanged).

In [ ]:
class BCEDiceLoss(nn.Module):
    def __init__(self, bce_weight=0.5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.bce_weight = bce_weight

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        probs = torch.sigmoid(logits)
        smooth = 1.0
        intersection = (probs * targets).sum()
        dice = (2.0 * intersection + smooth) / (probs.sum() + targets.sum() + smooth)
        dice_loss = 1.0 - dice
        return self.bce_weight * bce_loss + (1 - self.bce_weight) * dice_loss


def compute_metrics(preds, targets, threshold=0.5):
    """Compute precision, recall, F1, and IoU."""
    preds_bin = (preds > threshold).float()
    tp = (preds_bin * targets).sum().item()
    fp = (preds_bin * (1 - targets)).sum().item()
    fn = ((1 - preds_bin) * targets).sum().item()
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    iou = tp / (tp + fp + fn + 1e-8)
    return {"precision": precision, "recall": recall, "f1": f1, "iou": iou}

## 6. Training Loop

In [ ]:
import time
from tqdm.auto import tqdm

NUM_EPOCHS = 15  # fewer epochs for CPU training
LR = 3e-4       # slightly higher LR to converge faster

criterion = BCEDiceLoss(bce_weight=0.5)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

best_f1 = 0.0
history = {"train_loss": [], "val_loss": [], "val_f1": [], "val_iou": []}
train_start = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_start = time.time()

    # --- Train ---
    model.train()
    running_loss = 0.0
    for img_a, img_b, label in tqdm(train_dl, desc=f"Epoch {epoch}/{NUM_EPOCHS} [train]", leave=False):
        img_a = img_a.to(device)
        img_b = img_b.to(device)
        label = label.to(device)

        logits = model(img_a, img_b)
        loss = criterion(logits, label)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        running_loss += loss.item() * img_a.size(0)

    train_loss = running_loss / len(train_ds)
    scheduler.step()

    # --- Validate ---
    model.eval()
    val_loss_sum = 0.0
    all_preds, all_targets = [], []
    with torch.no_grad():
        for img_a, img_b, label in val_dl:
            img_a = img_a.to(device)
            img_b = img_b.to(device)
            label = label.to(device)

            logits = model(img_a, img_b)
            val_loss_sum += criterion(logits, label).item() * img_a.size(0)
            all_preds.append(torch.sigmoid(logits).cpu())
            all_targets.append(label.cpu())

    val_loss = val_loss_sum / len(val_ds)
    preds_cat = torch.cat(all_preds)
    targets_cat = torch.cat(all_targets)
    metrics = compute_metrics(preds_cat, targets_cat)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_f1"].append(metrics["f1"])
    history["val_iou"].append(metrics["iou"])

    elapsed_min = (time.time() - epoch_start) / 60
    total_min = (time.time() - train_start) / 60
    eta_min = elapsed_min * (NUM_EPOCHS - epoch)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"F1={metrics['f1']:.4f} | "
        f"IoU={metrics['iou']:.4f} | "
        f"P={metrics['precision']:.4f} R={metrics['recall']:.4f} | "
        f"{elapsed_min:.1f}min (ETA: {eta_min:.0f}min)"
    )

    if metrics["f1"] > best_f1:
        best_f1 = metrics["f1"]
        torch.save(model.state_dict(), "best_siamese_unet.pth")
        print(f"  >> Saved best model (F1={best_f1:.4f})")

total_time = (time.time() - train_start) / 60
print(f"\nTraining complete in {total_time:.1f} minutes. Best val F1: {best_f1:.4f}")

## 7. Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history["train_loss"], label="Train")
axes[0].plot(history["val_loss"], label="Val")
axes[0].set_title("Loss")
axes[0].legend()

axes[1].plot(history["val_f1"])
axes[1].set_title("Val F1 Score")

axes[2].plot(history["val_iou"])
axes[2].set_title("Val IoU")

for ax in axes:
    ax.set_xlabel("Epoch")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Evaluate on Test Set

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load("best_siamese_unet.pth", map_location=device))
model.eval()

all_preds, all_targets = [], []
with torch.no_grad():
    for img_a, img_b, label in tqdm(test_dl, desc="Testing"):
        logits = model(img_a.to(device), img_b.to(device))
        all_preds.append(torch.sigmoid(logits).cpu())
        all_targets.append(label)

preds = torch.cat(all_preds)
targets = torch.cat(all_targets)
test_metrics = compute_metrics(preds, targets)

print(f"\nTest Results:")
print(f"  F1 Score:  {test_metrics['f1']:.4f}")
print(f"  IoU:       {test_metrics['iou']:.4f}")
print(f"  Precision: {test_metrics['precision']:.4f}")
print(f"  Recall:    {test_metrics['recall']:.4f}")

## 9. Visualize Predictions

In [ ]:
MEAN = np.array([0.485, 0.456, 0.406])
STD = np.array([0.229, 0.224, 0.225])

def denorm(tensor):
    img = tensor.permute(1, 2, 0).numpy()
    img = img * STD + MEAN
    return np.clip(img, 0, 1)

fig, axes = plt.subplots(4, 4, figsize=(16, 16))
sample_indices = np.random.choice(len(test_ds), 4, replace=False)

for row, idx in enumerate(sample_indices):
    img_a, img_b, label = test_ds[idx]
    with torch.no_grad():
        logit = model(img_a.unsqueeze(0).to(device), img_b.unsqueeze(0).to(device))
        pred = (torch.sigmoid(logit) > 0.5).squeeze().cpu().numpy()

    axes[row, 0].imshow(denorm(img_a))
    axes[row, 0].set_title("Before")
    axes[row, 1].imshow(denorm(img_b))
    axes[row, 1].set_title("After")
    axes[row, 2].imshow(label.squeeze(), cmap="gray")
    axes[row, 2].set_title("Ground Truth")
    axes[row, 3].imshow(pred, cmap="gray")
    axes[row, 3].set_title("Prediction")

for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 10. Export Model for Deployment

Export as TorchScript for the web app. Download the `.pt` file and place it in
your app's `data/` folder, then set the environment variable:

```
CHANGE_MODEL_PATH=data/siamese_unet.pt
```

In [ ]:
model.eval()
model_cpu = model.cpu()

# Trace with example inputs
example_a = torch.randn(1, 3, 256, 256)
example_b = torch.randn(1, 3, 256, 256)
traced = torch.jit.trace(model_cpu, (example_a, example_b))

export_path = "siamese_unet.pt"
traced.save(export_path)
size_mb = os.path.getsize(export_path) / 1e6
print(f"Exported TorchScript model: {export_path} ({size_mb:.1f} MB)")
print("\nDownload this file and place it in your app's data/ directory.")
print('Then set: CHANGE_MODEL_PATH=data/siamese_unet.pt')

In [ ]:
# Quick sanity check: verify exported model produces same output
loaded = torch.jit.load(export_path)
with torch.no_grad():
    out_orig = model_cpu(example_a, example_b)
    out_loaded = loaded(example_a, example_b)
    diff = (out_orig - out_loaded).abs().max().item()
    print(f"Max diff between original and exported: {diff:.8f}")
    assert diff < 1e-5, "Export verification failed!"
    print("Export verified successfully.")

## 11. Download from Colab

Run this cell to trigger a browser download of the model file.

In [ ]:
try:
    from google.colab import files
    files.download("siamese_unet.pt")
    files.download("best_siamese_unet.pth")
except ImportError:
    print("Not running in Colab. Files saved locally:")
    print(f"  - {export_path}")
    print(f"  - best_siamese_unet.pth")